# 09 — Validate YOLO Silhouette ↔ Skeleton Alignment

এই notebook `08_extract_silhouettes_only.ipynb` শেষ হওয়ার পর run করবেন।

It maps:

```text
existing skeleton .npz
+
new YOLO silhouette .npz
```

by the common CASIA-B sample key:

```text
subject + condition + seq + view
```

Default expected paths:

```text
Skeleton root:
data/poses/<auto_detect_pose_tag>/

YOLO silhouette root:
data/silhouettes/yolo11m_seg_64x44/
```

Main outputs:

```text
data/fusion_index/pose_index.csv
data/fusion_index/silhouette_index.csv
data/fusion_index/alignment_report.csv
data/fusion_index/multimodal_index.csv

data/fusion_splits/train_LT_fusion.csv
data/fusion_splits/gallery_LT_fusion.csv
data/fusion_splits/probe_LT_nm_fusion.csv
data/fusion_splits/probe_LT_bg_fusion.csv
data/fusion_splits/probe_LT_cl_fusion.csv
```

Important:

- This notebook does **not** extract skeleton or silhouette again.
- It only validates and creates fusion-ready CSV files.
- For late fusion / adaptive gated fusion, exact frame-to-frame extraction in one script is not required.
- Fusion loader will use the common temporal range:

```python
T = min(T_pose, T_silhouette)
```

In [1]:
# ============================================================
# CELL 1 — Imports, paths, config
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXP_DIR = Path("/media/wadud/DriveUbuntu/GaitRecognition 2.0")

POSE_TAG = None
# None means auto-detect the largest pose folder under data/poses.
# Example manual value:
# POSE_TAG = "yolo26l_pose"

SILHOUETTE_TAG = "yolo11m_seg_64x44"

POSES_DIR = EXP_DIR / "data" / "poses"
SILHOUETTES_DIR = EXP_DIR / "data" / "silhouettes"

SPLIT_DIR = EXP_DIR / "data" / "splits"
FUSION_INDEX_DIR = EXP_DIR / "data" / "fusion_index"
FUSION_SPLIT_DIR = EXP_DIR / "data" / "fusion_splits"
REPORT_DIR = EXP_DIR / "data" / "reports"

for d in [FUSION_INDEX_DIR, FUSION_SPLIT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL = 124 * 10 * 11
EXPECTED_VIEWS = ["000", "018", "036", "054", "072", "090", "108", "126", "144", "162", "180"]

print("=" * 80)
print("YOLO Silhouette ↔ Skeleton Alignment Validation")
print("=" * 80)
print("EXP_DIR       :", EXP_DIR)
print("POSE_TAG      :", POSE_TAG)
print("SILHOUETTE_TAG:", SILHOUETTE_TAG)
print("=" * 80)

YOLO Silhouette ↔ Skeleton Alignment Validation
EXP_DIR       : /media/wadud/DriveUbuntu/GaitRecognition 2.0
POSE_TAG      : None
SILHOUETTE_TAG: yolo11m_seg_64x44


In [2]:
# ============================================================
# CELL 2 — Auto-detect pose root and silhouette root
# ============================================================

def count_npz_files(root: Path):
    if not root.exists():
        return 0
    return len(list(root.rglob("*.npz")))

def detect_largest_child(root: Path, tag=None, suffix="*.npz"):
    if tag is not None:
        p = root / tag
        if not p.exists():
            raise FileNotFoundError(f"Requested folder does not exist: {p}")
        return p

    if not root.exists():
        raise FileNotFoundError(f"Root folder does not exist: {root}")

    candidates = [p for p in root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No subfolders found under: {root}")

    counts = []
    for p in candidates:
        counts.append((len(list(p.rglob(suffix))), p))

    counts.sort(reverse=True, key=lambda x: x[0])

    print("Candidate folders under", root)
    for count, p in counts:
        print(f"  {p.name}: {count} .npz files")

    return counts[0][1]

POSE_ROOT = detect_largest_child(POSES_DIR, POSE_TAG)
SIL_ROOT = detect_largest_child(SILHOUETTES_DIR, SILHOUETTE_TAG)

print()
print("Selected POSE_ROOT:", POSE_ROOT)
print("Selected SIL_ROOT :", SIL_ROOT)
print("Pose files        :", count_npz_files(POSE_ROOT))
print("Silhouette files  :", count_npz_files(SIL_ROOT))
print("Expected total    :", EXPECTED_TOTAL)

assert POSE_ROOT.exists(), f"POSE_ROOT does not exist: {POSE_ROOT}"
assert SIL_ROOT.exists(), f"SIL_ROOT does not exist: {SIL_ROOT}"

Candidate folders under /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/poses
  yolo26l_pose: 13640 .npz files

Selected POSE_ROOT: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/poses/yolo26l_pose
Selected SIL_ROOT : /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/silhouettes/yolo11m_seg_64x44
Pose files        : 13640
Silhouette files  : 13640
Expected total    : 13640


In [3]:
# ============================================================
# CELL 3 — Build pose and silhouette indexes
# ============================================================

def record_from_modal_path(path: Path, root: Path, modal_name: str):
    rel = path.relative_to(root)

    if len(rel.parts) < 3:
        raise ValueError(f"Unexpected {modal_name} path structure: {path}")

    subject = str(rel.parts[0])
    condition_seq = str(rel.parts[1])
    view = str(path.stem)

    if "-" not in condition_seq:
        raise ValueError(f"Unexpected condition-seq folder: {condition_seq}")

    condition, seq = condition_seq.split("-", 1)

    return {
        "subject": subject,
        "condition": condition,
        "seq": seq,
        "view": view,
        "condition_seq": condition_seq,
        "key": f"{subject}-{condition}-{seq}-{view}",
        f"{modal_name}_path": str(path),
    }

pose_files = sorted(POSE_ROOT.rglob("*.npz"))
sil_files = sorted(SIL_ROOT.rglob("*.npz"))

pose_records = []
sil_records = []

bad_pose = []
bad_sil = []

for p in pose_files:
    try:
        pose_records.append(record_from_modal_path(p, POSE_ROOT, "pose"))
    except Exception as e:
        bad_pose.append({"path": str(p), "error": str(e)})

for p in sil_files:
    try:
        sil_records.append(record_from_modal_path(p, SIL_ROOT, "silhouette"))
    except Exception as e:
        bad_sil.append({"path": str(p), "error": str(e)})

df_pose = pd.DataFrame(pose_records)
df_sil = pd.DataFrame(sil_records)

pose_index_csv = FUSION_INDEX_DIR / "pose_index.csv"
sil_index_csv = FUSION_INDEX_DIR / "silhouette_index.csv"

df_pose.to_csv(pose_index_csv, index=False)
df_sil.to_csv(sil_index_csv, index=False)

if bad_pose:
    pd.DataFrame(bad_pose).to_csv(FUSION_INDEX_DIR / "bad_pose_paths.csv", index=False)
if bad_sil:
    pd.DataFrame(bad_sil).to_csv(FUSION_INDEX_DIR / "bad_silhouette_paths.csv", index=False)

print("Pose index rows      :", len(df_pose))
print("Silhouette index rows:", len(df_sil))
print("Bad pose paths       :", len(bad_pose))
print("Bad silhouette paths :", len(bad_sil))
print("Saved:", pose_index_csv)
print("Saved:", sil_index_csv)

display(df_pose.head())
display(df_sil.head())

Pose index rows      : 13640
Silhouette index rows: 13640
Bad pose paths       : 0
Bad silhouette paths : 0
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/fusion_index/pose_index.csv
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/fusion_index/silhouette_index.csv


,subject,condition,seq,view,condition_seq,key,pose_path
0,001,bg,01,000,bg-01,001-bg-01-000,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
1,001,bg,01,018,bg-01,001-bg-01-018,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
2,001,bg,01,036,bg-01,001-bg-01-036,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
3,001,bg,01,054,bg-01,001-bg-01-054,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
4,001,bg,01,072,bg-01,001-bg-01-072,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...


,subject,condition,seq,view,condition_seq,key,silhouette_path
0,001,bg,01,000,bg-01,001-bg-01-000,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
1,001,bg,01,018,bg-01,001-bg-01-018,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
2,001,bg,01,036,bg-01,001-bg-01-036,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
3,001,bg,01,054,bg-01,001-bg-01-054,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...
4,001,bg,01,072,bg-01,001-bg-01-072,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...


In [4]:
# ============================================================
# CELL 4 — Check duplicate keys
# ============================================================

def check_duplicate_keys(df, name):
    dup = df[df.duplicated("key", keep=False)].sort_values("key")
    print(f"{name} duplicate keys:", len(dup))

    if len(dup) > 0:
        dup_csv = FUSION_INDEX_DIR / f"{name}_duplicate_keys.csv"
        dup.to_csv(dup_csv, index=False)
        print("Saved duplicate report:", dup_csv)
        display(dup.head(20))

check_duplicate_keys(df_pose, "pose")
check_duplicate_keys(df_sil, "silhouette")

assert not df_pose.duplicated("key").any(), "Pose index has duplicate sample keys."
assert not df_sil.duplicated("key").any(), "Silhouette index has duplicate sample keys."

pose duplicate keys: 0
silhouette duplicate keys: 0


In [5]:
# ============================================================
# CELL 5 — Match skeleton and silhouette by sample key
# ============================================================

merge_keys = ["subject", "condition", "seq", "view", "condition_seq", "key"]

df_all = pd.merge(
    df_pose,
    df_sil,
    on=merge_keys,
    how="outer",
    indicator=True,
)

df_matched = df_all[df_all["_merge"] == "both"].copy()
df_pose_only = df_all[df_all["_merge"] == "left_only"].copy()
df_sil_only = df_all[df_all["_merge"] == "right_only"].copy()

missing_modalities_csv = FUSION_INDEX_DIR / "missing_modalities_report.csv"
df_all.to_csv(missing_modalities_csv, index=False)

print("Matched samples :", len(df_matched))
print("Pose only       :", len(df_pose_only))
print("Silhouette only :", len(df_sil_only))
print("Expected total  :", EXPECTED_TOTAL)
print("Saved:", missing_modalities_csv)

if len(df_pose_only) > 0:
    print("\nPose-only examples:")
    display(df_pose_only.head(20))

if len(df_sil_only) > 0:
    print("\nSilhouette-only examples:")
    display(df_sil_only.head(20))

Matched samples : 13640
Pose only       : 0
Silhouette only : 0
Expected total  : 13640
Saved: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/fusion_index/missing_modalities_report.csv


In [6]:
# ============================================================
# CELL 6 — Read metadata: frame counts and quality
# ============================================================

def scalar_to_str(x):
    try:
        if isinstance(x, np.ndarray):
            if x.shape == ():
                return str(x.item())
            if len(x) == 1:
                return str(x[0])
        return str(x)
    except Exception:
        return str(x)

def read_pose_meta(path):
    try:
        data = np.load(path)
        if "keypoints_norm_filled" not in data.files:
            return {
                "T_pose": 0,
                "pose_shape": "",
                "pose_error": "missing keypoints_norm_filled",
            }

        X = data["keypoints_norm_filled"]
        return {
            "T_pose": int(X.shape[0]),
            "pose_shape": str(tuple(X.shape)),
            "pose_error": "",
        }
    except Exception as e:
        return {
            "T_pose": 0,
            "pose_shape": "",
            "pose_error": str(e),
        }

def read_sil_meta(path):
    try:
        data = np.load(path)
        if "silhouettes" not in data.files:
            return {
                "T_silhouette": 0,
                "silhouette_shape": "",
                "sil_valid_frame_ratio": 0.0,
                "sil_mean_det_score": 0.0,
                "sil_mean_mask_area": 0.0,
                "silhouette_error": "missing silhouettes",
            }

        S = data["silhouettes"]
        T = int(S.shape[0])

        if "frame_valid" in data.files:
            frame_valid = data["frame_valid"].astype(bool)
        else:
            frame_valid = np.ones((T,), dtype=bool)

        if "det_scores" in data.files:
            det_scores = data["det_scores"].astype(np.float32)
            mean_det_score = float(np.nanmean(det_scores)) if len(det_scores) else 0.0
        else:
            mean_det_score = np.nan

        if "mask_areas" in data.files:
            mask_areas = data["mask_areas"].astype(np.float32)
            mean_mask_area = float(np.nanmean(mask_areas)) if len(mask_areas) else 0.0
        elif "foreground_areas" in data.files:
            mask_areas = data["foreground_areas"].astype(np.float32)
            mean_mask_area = float(np.nanmean(mask_areas)) if len(mask_areas) else 0.0
        else:
            mean_mask_area = float((S > 0).sum(axis=(1, 2)).mean()) if len(S) else 0.0

        return {
            "T_silhouette": T,
            "silhouette_shape": str(tuple(S.shape)),
            "sil_valid_frame_ratio": float(frame_valid.mean()) if len(frame_valid) else 0.0,
            "sil_mean_det_score": mean_det_score,
            "sil_mean_mask_area": mean_mask_area,
            "silhouette_error": "",
        }

    except Exception as e:
        return {
            "T_silhouette": 0,
            "silhouette_shape": "",
            "sil_valid_frame_ratio": 0.0,
            "sil_mean_det_score": 0.0,
            "sil_mean_mask_area": 0.0,
            "silhouette_error": str(e),
        }

rows = []

for _, row in tqdm(df_matched.iterrows(), total=len(df_matched), desc="Reading pose/silhouette metadata"):
    pose_meta = read_pose_meta(row["pose_path"])
    sil_meta = read_sil_meta(row["silhouette_path"])

    T_pose = int(pose_meta["T_pose"])
    T_sil = int(sil_meta["T_silhouette"])

    T_common = min(T_pose, T_sil)
    T_diff = abs(T_pose - T_sil)

    if T_pose == 0 or T_sil == 0:
        status = "read_error"
    elif T_diff == 0:
        status = "exact"
    elif T_diff <= 2:
        status = "small_mismatch"
    else:
        status = "large_mismatch"

    rows.append({
        "key": row["key"],
        "subject": row["subject"],
        "condition": row["condition"],
        "seq": row["seq"],
        "view": row["view"],
        "condition_seq": row["condition_seq"],
        "pose_path": row["pose_path"],
        "silhouette_path": row["silhouette_path"],
        **pose_meta,
        **sil_meta,
        "T_common": T_common,
        "T_diff": T_diff,
        "alignment_status": status,
    })

df_align = pd.DataFrame(rows)

alignment_csv = FUSION_INDEX_DIR / "alignment_report.csv"
df_align.to_csv(alignment_csv, index=False)

print("Saved alignment report:", alignment_csv)
print("\nAlignment status:")
display(df_align["alignment_status"].value_counts())

print("\nFrame and quality summary:")
display(df_align[[
    "T_pose",
    "T_silhouette",
    "T_common",
    "T_diff",
    "sil_valid_frame_ratio",
    "sil_mean_det_score",
    "sil_mean_mask_area",
]].describe())

display(df_align.head())

NameError: name 'tqdm' is not defined

In [ ]:
# ============================================================
# CELL 7 — Create clean multimodal index
# ============================================================

# For late fusion, exact/small/large mismatch are usable as long as T_common is enough.
# The fusion loader will use T = min(T_pose, T_silhouette).
SAFE_STATUSES = {"exact", "small_mismatch", "large_mismatch"}

MIN_COMMON_FRAMES = 20
MIN_VALID_SIL_RATIO = 0.30

df_multi = df_align[df_align["alignment_status"].isin(SAFE_STATUSES)].copy()
df_multi = df_multi[df_multi["T_common"] >= MIN_COMMON_FRAMES].copy()
df_multi = df_multi[df_multi["sil_valid_frame_ratio"] >= MIN_VALID_SIL_RATIO].copy()

multimodal_csv = FUSION_INDEX_DIR / "multimodal_index.csv"
df_multi.to_csv(multimodal_csv, index=False)

print("Usable multimodal samples:", len(df_multi))
print("Expected total           :", EXPECTED_TOTAL)
print("Dropped from matched     :", len(df_align) - len(df_multi))
print("Saved:", multimodal_csv)

print("\nUsable alignment status:")
display(df_multi["alignment_status"].value_counts())

if len(df_multi) < EXPECTED_TOTAL:
    missing_keys = set(df_pose["key"]) - set(df_multi["key"])
    df_missing_usable = df_pose[df_pose["key"].isin(missing_keys)].copy()
    missing_usable_csv = FUSION_INDEX_DIR / "not_usable_multimodal_keys.csv"
    df_missing_usable.to_csv(missing_usable_csv, index=False)
    print("Saved not-usable multimodal keys:", missing_usable_csv)
    display(df_missing_usable.head(20))

In [ ]:
# ============================================================
# CELL 8 — Condition/view count checks
# ============================================================

print("Counts by condition-seq:")
display(
    df_multi
    .groupby(["condition", "seq"])
    .size()
    .reset_index(name="count")
    .sort_values(["condition", "seq"])
)

print("Counts by view:")
display(
    df_multi
    .groupby("view")
    .size()
    .reset_index(name="count")
    .sort_values("view")
)

print("Counts by condition and view:")
display(
    df_multi
    .groupby(["condition", "view"])
    .size()
    .reset_index(name="count")
    .sort_values(["condition", "view"])
)

In [ ]:
# ============================================================
# CELL 9 — Create fusion split CSV files
# ============================================================

split_files = [
    "train_ST.csv", "test_ST.csv", "gallery_ST.csv", "probe_ST_nm.csv", "probe_ST_bg.csv", "probe_ST_cl.csv",
    "train_MT.csv", "test_MT.csv", "gallery_MT.csv", "probe_MT_nm.csv", "probe_MT_bg.csv", "probe_MT_cl.csv",
    "train_LT.csv", "test_LT.csv", "gallery_LT.csv", "probe_LT_nm.csv", "probe_LT_bg.csv", "probe_LT_cl.csv",
]

csv_dtype = {
    "pose_path": str,
    "subject": str,
    "condition": str,
    "seq": str,
    "view": str,
}

lookup_cols = [
    "subject", "condition", "seq", "view",
    "pose_path", "silhouette_path",
    "T_pose", "T_silhouette", "T_common", "T_diff",
    "alignment_status",
    "sil_valid_frame_ratio",
    "sil_mean_det_score",
    "sil_mean_mask_area",
]

df_lookup = df_multi[lookup_cols].copy()

split_summary_rows = []
created_paths = []

for filename in split_files:
    src = SPLIT_DIR / filename

    if not src.exists():
        print("[SKIP missing]", src)
        continue

    df_split = pd.read_csv(src, dtype=csv_dtype)

    # Merge by CASIA-B sample key columns. Drop old pose_path because df_lookup has verified pose path.
    df_merge = pd.merge(
        df_split.drop(columns=["pose_path"], errors="ignore"),
        df_lookup,
        on=["subject", "condition", "seq", "view"],
        how="left",
        indicator=True,
    )

    df_fusion = df_merge[df_merge["_merge"] == "both"].drop(columns=["_merge"]).copy()
    df_missing = df_merge[df_merge["_merge"] != "both"].drop(columns=["_merge"]).copy()

    out_name = filename.replace(".csv", "_fusion.csv")
    out_path = FUSION_SPLIT_DIR / out_name
    df_fusion.to_csv(out_path, index=False)
    created_paths.append(out_path)

    missing_path = None
    if len(df_missing) > 0:
        missing_path = FUSION_SPLIT_DIR / filename.replace(".csv", "_missing_multimodal.csv")
        df_missing.to_csv(missing_path, index=False)

    split_summary_rows.append({
        "source_split": filename,
        "source_rows": len(df_split),
        "fusion_rows": len(df_fusion),
        "missing_rows": len(df_missing),
        "out_csv": str(out_path),
        "missing_csv": str(missing_path) if missing_path else "",
    })

df_split_summary = pd.DataFrame(split_summary_rows)
split_summary_csv = FUSION_SPLIT_DIR / "fusion_split_summary.csv"
df_split_summary.to_csv(split_summary_csv, index=False)

print("Fusion split summary:")
display(df_split_summary)
print("Saved:", split_summary_csv)

print("\nCreated fusion split files:")
for p in created_paths:
    print("-", p)

In [ ]:
# ============================================================
# CELL 10 — LT split expected count check
# ============================================================

expected_lt = {
    "train_LT_fusion.csv": 8140,
    "test_LT_fusion.csv": 5500,
    "gallery_LT_fusion.csv": 2200,
    "probe_LT_nm_fusion.csv": 1100,
    "probe_LT_bg_fusion.csv": 1100,
    "probe_LT_cl_fusion.csv": 1100,
}

for fname, expected in expected_lt.items():
    p = FUSION_SPLIT_DIR / fname
    if p.exists():
        df = pd.read_csv(p, dtype=str)
        print(f"{fname}: rows={len(df)}, expected={expected}, subjects={df['subject'].nunique()}")
    else:
        print("[MISSING]", fname)

In [ ]:
# ============================================================
# CELL 11 — Visual check: silhouette and skeleton same sample
# ============================================================

COCO_EDGES = [
    (0, 1), (0, 2),
    (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]

def plot_sample_alignment(row, num_frames=6):
    pose_data = np.load(row["pose_path"])
    sil_data = np.load(row["silhouette_path"])

    X = pose_data["keypoints_norm_filled"].astype(np.float32)
    S = sil_data["silhouettes"].astype(np.uint8)

    T = min(len(X), len(S))
    if T <= 0:
        print("Empty sample:", row["key"])
        return

    frame_ids = np.linspace(0, T - 1, num=min(num_frames, T)).astype(int)

    fig, axes = plt.subplots(2, len(frame_ids), figsize=(2.5 * len(frame_ids), 5.5))

    if len(frame_ids) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    for i, t in enumerate(frame_ids):
        ax0 = axes[0, i]
        ax0.imshow(S[t], cmap="gray")
        ax0.set_title(f"sil t={t}")
        ax0.axis("off")

        pts = X[t]
        ax1 = axes[1, i]

        for a, b in COCO_EDGES:
            ax1.plot([pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]], linewidth=2)

        ax1.scatter(pts[:, 0], pts[:, 1], s=10)
        ax1.set_xlim(0, 1)
        ax1.set_ylim(1, 0)
        ax1.set_aspect("equal")
        ax1.set_title(f"skel t={t}")
        ax1.axis("off")

    fig.suptitle(
        f"{row['key']} | T_pose={row['T_pose']} T_sil={row['T_silhouette']} status={row['alignment_status']}"
    )
    fig.tight_layout()
    plt.show()

if len(df_multi) == 0:
    print("No multimodal samples found.")
else:
    sample_row = df_multi.sample(1, random_state=42).iloc[0]
    plot_sample_alignment(sample_row, num_frames=6)

In [ ]:
# ============================================================
# CELL 12 — Fusion loader dry-run
# ============================================================

def crop_pair_for_fusion(pose_path, silhouette_path, seq_len=60, random_crop=True):
    pose_data = np.load(pose_path)
    sil_data = np.load(silhouette_path)

    X = pose_data["keypoints_norm_filled"].astype(np.float32)
    S = sil_data["silhouettes"].astype(np.uint8)

    T = min(len(X), len(S))
    if T <= 0:
        raise ValueError("Empty pose/silhouette sequence.")

    X = X[:T]
    S = S[:T]

    if T >= seq_len:
        if random_crop:
            start = np.random.randint(0, T - seq_len + 1)
        else:
            start = max(0, (T - seq_len) // 2)

        X_clip = X[start:start + seq_len]
        S_clip = S[start:start + seq_len]
    else:
        pad_len = seq_len - T
        X_pad = np.repeat(X[-1:], pad_len, axis=0)
        S_pad = np.repeat(S[-1:], pad_len, axis=0)

        X_clip = np.concatenate([X, X_pad], axis=0)
        S_clip = np.concatenate([S, S_pad], axis=0)

    return X_clip, S_clip

lt_train_fusion = FUSION_SPLIT_DIR / "train_LT_fusion.csv"

if lt_train_fusion.exists():
    df_train_fusion = pd.read_csv(lt_train_fusion, dtype=str)
    row = df_train_fusion.iloc[0]

    X_clip, S_clip = crop_pair_for_fusion(
        row["pose_path"],
        row["silhouette_path"],
        seq_len=60,
        random_crop=False,
    )

    print("Fusion train rows:", len(df_train_fusion))
    print("X_clip shape     :", X_clip.shape)
    print("S_clip shape     :", S_clip.shape)
    print("Subject          :", row["subject"])
    print("Condition        :", row["condition"], row["seq"], row["view"])

    assert X_clip.shape == (60, 17, 2)
    assert S_clip.shape == (60, 64, 44)
    print("[OK] Fusion loader dry-run passed.")
else:
    print("Missing:", lt_train_fusion)

In [ ]:
# ============================================================
# CELL 13 — Save final summary JSON
# ============================================================

summary = {
    "exp_dir": str(EXP_DIR),
    "pose_root": str(POSE_ROOT),
    "silhouette_root": str(SIL_ROOT),
    "silhouette_tag": SILHOUETTE_TAG,
    "num_pose_files": int(len(df_pose)),
    "num_silhouette_files": int(len(df_sil)),
    "num_matched": int(len(df_matched)),
    "num_pose_only": int(len(df_pose_only)),
    "num_silhouette_only": int(len(df_sil_only)),
    "num_alignment_rows": int(len(df_align)),
    "num_multimodal_usable": int(len(df_multi)),
    "alignment_status_counts": df_align["alignment_status"].value_counts().to_dict(),
    "multimodal_index_csv": str(multimodal_csv),
    "alignment_report_csv": str(alignment_csv),
    "fusion_split_dir": str(FUSION_SPLIT_DIR),
    "fusion_split_summary_csv": str(split_summary_csv),
}

summary_path = FUSION_INDEX_DIR / "fusion_alignment_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 80)
print("FUSION ALIGNMENT SUMMARY")
print("=" * 80)
for k, v in summary.items():
    print(k, ":", v)
print("=" * 80)
print("Saved:", summary_path)

## After this notebook

If the final checks show:

```text
Pose files        : 13640
Silhouette files  : 13640
Matched samples   : 13640
Pose only         : 0
Silhouette only   : 0
train_LT_fusion   : 8140
gallery_LT_fusion : 2200
probe_LT_*_fusion : 1100 each
```

then the multimodal dataset is ready.

Next script after this:

```text
10_train_silhouette_baseline.ipynb
```

That script will train the silhouette-only expert before fusion.